In [1]:
import pandas as pd
import re

train = pd.read_json("D:/vuln-detection-ml/data/train.jsonl", lines=True)
valid = pd.read_json("D:/vuln-detection-ml/data/valid.jsonl", lines=True)
def clean_code(code):
    code = re.sub(r"/\*.*?\*/", " ", code, flags=re.DOTALL)  # blocks comments
    code = re.sub(r"//[^\n]*", " ", code)                    # line comments
    code = re.sub(r"\s+", " ", code)                         # collapse whitespace
    return code.strip()

train["clean"] = train["func"].apply(clean_code)
valid["clean"] = valid["func"].apply(clean_code)

print(train["clean"].iloc[0][:300])

bool virtio_scsi_handle_cmd_req_prepare(VirtIOSCSI *s, VirtIOSCSIReq *req) { VirtIOSCSICommon *vs = &s->parent_obj; SCSIDevice *d; int rc; rc = virtio_scsi_parse_req(req, sizeof(VirtIOSCSICmdReq) + vs->cdb_size, sizeof(VirtIOSCSICmdResp) + vs->sense_size); if (rc < 0) { if (rc == -ENOTSUP) { virtio_


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    token_pattern=r"[A-Za-z_][A-Za-z0-9_]*|\d+|==|!=|<=|>=|->|&&|\|\||[^\s\w]",
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
)

X_train = vectorizer.fit_transform(train["clean"])   # fit ONLY on train
X_valid = vectorizer.transform(valid["clean"])       # just transform here
y_train, y_valid = train["target"], valid["target"]

print(X_train.shape)

(21854, 50000)


In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Dummy accuracy:", dummy.score(X_valid, y_valid))

nb = MultinomialNB().fit(X_train, y_train)
pred = nb.predict(X_valid)
print(classification_report(y_valid, pred, digits=3))
print(confusion_matrix(y_valid, pred))

Dummy accuracy: 0.5439238653001464
              precision    recall  f1-score   support

           0      0.628     0.745     0.682      1486
           1      0.609     0.474     0.533      1246

    accuracy                          0.622      2732
   macro avg      0.619     0.610     0.608      2732
weighted avg      0.620     0.622     0.614      2732

[[1107  379]
 [ 655  591]]
